# 17_mamba_sequence_baseline_subset.ipynb

Ελεγχόμενο notebook-σκελετός για πειράματα Mamba σε μικρό, σταθερό υποσύνολο αιολικών πάρκων.

Το notebook χρησιμοποιεί αποκλειστικά τα canonical split artifacts:

- `data/processed/train_final.csv`
- `data/processed/val_final.csv`
- `data/processed/test_final.csv`

Όρια και κανόνες ασφάλειας:

- Χρησιμοποιούνται μόνο τα σταθερά πάρκα `00183`, `00198`, `00303`, `00427`.
- Σε αυτό το branch υπάρχει μόνο Mamba-only sequence baseline scaffold.
- Το `data/processed/baseline_metrics.csv` δεν ενημερώνεται.
- Δεν γράφονται checkpoints ή model binaries.
- Δεν γράφονται generated CSV outputs εκτός αν οριστεί ρητά `EXPORT_RESULTS = True`.
- Οποιαδήποτε προαιρετικά exports παραμένουν τοπικά κάτω από `data/processed/diagnostics/nn_sequence_subset_mamba/`.
- Η προεπιλεγμένη εκτέλεση δεν είναι πλήρες πείραμα. Τυχόν μικρή εκπαίδευση είναι μόνο smoke validation και όχι τεκμήριο manuscript.
- Τα αποτελέσματα του υποσυνόλου δεν αντικαθιστούν το benchmark και δεν τεκμηριώνουν ισχυρισμό βελτίωσης ή ισοδυναμίας με το `baseline_metrics.csv`.


## Συμβόλαιο Στόχου Ακολουθίας

Για κάθε split, με ανεξάρτητη επεξεργασία ανά park:

1. Οι γραμμές ταξινομούνται κατά `park_id` και `timestamp`.
2. Η χρονοσειρά διασπάται σε συνεχόμενα segments, με διακοπή σε timestamp gaps.
3. Με `LOOKBACK_STEPS = 24`, κάθε δείγμα ορίζεται ως:
   - `X = feature rows [i, ..., i+23]`
   - `y = target at row i+24`
4. Τα features της χρονικής στιγμής του προβλεπόμενου στόχου δεν μπαίνουν ποτέ στο `X`.
5. Δεν δημιουργούνται windows που περνούν όρια park, split ή timestamp gap.
6. Segments με λιγότερες από `LOOKBACK_STEPS + 1` γραμμές δεν παράγουν windows.


In [ ]:
# ============================================================
# NB17 | Imports, αναπαραγωγιμότητα και σταθερές
# ============================================================

from __future__ import annotations

import copy
import random
import sys
import time
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

from IPython.display import display

from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset


SEED = 42
SELECTED_PARKS = ["00183", "00198", "00303", "00427"]

TARGET_COLUMN = "Power_Output_Normalized"
PARK_ID_COLUMN = "park_id"
TIMESTAMP_COLUMN = "timestamp"
TEST_FLAG_COLUMN = "test_flag"
BASELINE_COLUMN = "Baseline_Prediction"
TURBINE_COLUMN = "turbine"

EXCLUDED_COLUMNS = {
    TARGET_COLUMN,
    PARK_ID_COLUMN,
    TIMESTAMP_COLUMN,
    TEST_FLAG_COLUMN,
    BASELINE_COLUMN,
    TURBINE_COLUMN,
}

LOOKBACK_STEPS = 24
EXPECTED_FREQ = pd.Timedelta(hours=1)
EXPECTED_NUMERIC_FEATURES = 41

# Προεπιλογές πλήρους run. Το notebook δεν τρέχει πλήρες πείραμα από προεπιλογή.
BATCH_SIZE = 512
D_MODEL = 32
MAMBA_LAYERS = 1
MAMBA_D_STATE = 16
MAMBA_D_CONV = 4
MAMBA_EXPAND = 2
DROPOUT = 0.0
EPOCHS = 30
PATIENCE = 5
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5

# Το smoke mode κρατά την προαιρετική δοκιμή μικρή και μη αποδεικτική για manuscript.
SMOKE_MODE = True
RUN_SMOKE_TRAINING = False
RUN_TEST_EVALUATION = False
SMOKE_EPOCHS = 2
SMOKE_PATIENCE = 1
SMOKE_MAX_WINDOWS = {
    "train": 512,
    "validation": 256,
    "test": 256,
}

# Τα προαιρετικά τοπικά exports είναι απενεργοποιημένα από προεπιλογή.
EXPORT_RESULTS = False


def set_reproducibility(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


set_reproducibility(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("SEED:", SEED)
print("DEVICE:", DEVICE)
print("SMOKE_MODE:", SMOKE_MODE)
print("RUN_SMOKE_TRAINING:", RUN_SMOKE_TRAINING)
print("RUN_TEST_EVALUATION:", RUN_TEST_EVALUATION)
print("EXPORT_RESULTS:", EXPORT_RESULTS)


## Προαιρετική εγκατάσταση Mamba σε Colab

Το notebook δεν εγκαθιστά packages αυτόματα. Για μελλοντικό Colab/GPU smoke run, εκτελέστε χειροκίνητα το dependency πριν από το import cell:

```bash
pip install "mamba-ssm[causal-conv1d]" --no-build-isolation
```

Η εγκατάσταση παραμένει εκτός του κανονικού local execution path ώστε το notebook να μείνει ασφαλές ως scaffold.


In [ ]:
# ============================================================
# NB17 | Έλεγχος περιβάλλοντος και device
# ============================================================

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA διαθέσιμο:", torch.cuda.is_available())
print("DEVICE:", DEVICE)

if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name(0))
else:
    print("Δεν υπάρχει GPU· το notebook scaffold παραμένει ασφαλές και δεν εκπαιδεύει από προεπιλογή.")


In [ ]:
# ============================================================
# NB17 | Paths και προαιρετικές σταθερές εξόδων
# ============================================================

def find_project_root(start_path: Path) -> Path:
    current = start_path.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError("Δεν βρέθηκε project root με φακέλους data/ και notebooks/.")


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

TRAIN_PATH = DATA_PROCESSED / "train_final.csv"
VAL_PATH = DATA_PROCESSED / "val_final.csv"
TEST_PATH = DATA_PROCESSED / "test_final.csv"
BASELINE_METRICS_PATH = DATA_PROCESSED / "baseline_metrics.csv"

# Προαιρετικός φάκελος εξόδων: local-only diagnostics.
OUTPUT_DIR = DATA_PROCESSED / "diagnostics" / "nn_sequence_subset_mamba"
EXPECTED_OUTPUT_DIR = OUTPUT_DIR

RUN_MANIFEST_PATH = OUTPUT_DIR / "nn_sequence_subset_mamba_run_manifest.csv"
WINDOW_AUDIT_PATH = OUTPUT_DIR / "nn_sequence_subset_mamba_window_audit.csv"
TRAINING_HISTORY_PATH = OUTPUT_DIR / "nn_sequence_subset_mamba_training_history.csv"
VALIDATION_METRICS_PATH = OUTPUT_DIR / "nn_sequence_subset_mamba_validation_metrics.csv"
SELECTED_TEST_METRICS_PATH = OUTPUT_DIR / "nn_sequence_subset_mamba_selected_test_metrics.csv"
TEST_PREDICTIONS_RESIDUALS_PATH = OUTPUT_DIR / "nn_sequence_subset_mamba_test_predictions_residuals.csv"

for label, path in [
    ("train", TRAIN_PATH),
    ("validation", VAL_PATH),
    ("test", TEST_PATH),
]:
    if not path.exists():
        raise FileNotFoundError(f"Λείπει το απαιτούμενο {label} split: {path}")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("TRAIN_PATH:", TRAIN_PATH)
print("VAL_PATH:", VAL_PATH)
print("TEST_PATH:", TEST_PATH)
print("Προαιρετικός φάκελος εξόδων:", OUTPUT_DIR)
print("Το baseline metrics path είναι μόνο για ανάγνωση στο NB17:", BASELINE_METRICS_PATH)


## Φόρτωση Και Έλεγχος Canonical Splits

Ο loader διαβάζει μόνο τις γραμμές των σταθερά επιλεγμένων parks. Το `park_id` διατηρείται ως zero-padded string και το `timestamp` γίνεται parse πριν από ταξινόμηση ή κατασκευή ακολουθιών.


In [ ]:
# ============================================================
# NB17 | Φόρτωση επιλεγμένων parks από τα canonical split artifacts
# ============================================================

REQUIRED_COLUMNS = {
    PARK_ID_COLUMN,
    TIMESTAMP_COLUMN,
    TEST_FLAG_COLUMN,
    TARGET_COLUMN,
    BASELINE_COLUMN,
}


def normalize_park_id(series: pd.Series) -> pd.Series:
    return series.astype(str).str.replace(".0", "", regex=False).str.zfill(5)


def read_filtered_split(path: Path, split_name: str, selected_parks: list[str]) -> pd.DataFrame:
    header = pd.read_csv(path, nrows=0).columns
    missing = sorted(REQUIRED_COLUMNS - set(header))
    if missing:
        raise KeyError(f"Λείπουν required columns από το {split_name}: {missing}")

    selected = set(selected_parks)
    frames: list[pd.DataFrame] = []
    for chunk in pd.read_csv(path, dtype={PARK_ID_COLUMN: "string"}, chunksize=100_000):
        chunk[PARK_ID_COLUMN] = normalize_park_id(chunk[PARK_ID_COLUMN])
        filtered = chunk.loc[chunk[PARK_ID_COLUMN].isin(selected)].copy()
        if not filtered.empty:
            frames.append(filtered)

    if not frames:
        raise ValueError(f"Δεν βρέθηκαν γραμμές στο {split_name} για τα selected parks: {selected_parks}")

    df = pd.concat(frames, ignore_index=True)
    df[TIMESTAMP_COLUMN] = pd.to_datetime(df[TIMESTAMP_COLUMN], errors="raise")
    df[PARK_ID_COLUMN] = normalize_park_id(df[PARK_ID_COLUMN])
    return df


train_raw = read_filtered_split(TRAIN_PATH, "train", SELECTED_PARKS)
val_raw = read_filtered_split(VAL_PATH, "validation", SELECTED_PARKS)
test_raw = read_filtered_split(TEST_PATH, "test", SELECTED_PARKS)

split_frames = {
    "train": train_raw,
    "validation": val_raw,
    "test": test_raw,
}

for split_name, df in split_frames.items():
    observed_parks = sorted(df[PARK_ID_COLUMN].unique().tolist())
    if observed_parks != SELECTED_PARKS:
        raise ValueError(f"Μη αναμενόμενα parks στο {split_name}: {observed_parks}")
    if df.duplicated(subset=[PARK_ID_COLUMN, TIMESTAMP_COLUMN]).any():
        raise ValueError(f"Βρέθηκαν duplicate (park_id, timestamp) rows στο {split_name}.")
    core_nulls = int(df[[PARK_ID_COLUMN, TIMESTAMP_COLUMN, TARGET_COLUMN]].isnull().sum().sum())
    if core_nulls:
        raise ValueError(f"Βρέθηκαν null values στις core {split_name} columns: {core_nulls}")

assert set(train_raw[TEST_FLAG_COLUMN].dropna().astype(int).unique()) <= {0}
assert set(val_raw[TEST_FLAG_COLUMN].dropna().astype(int).unique()) <= {0}
assert set(test_raw[TEST_FLAG_COLUMN].dropna().astype(int).unique()) <= {1}

split_summary_df = pd.DataFrame(
    [
        {
            "split": split_name,
            "rows": len(df),
            "parks": df[PARK_ID_COLUMN].nunique(),
            "min_timestamp": df[TIMESTAMP_COLUMN].min(),
            "max_timestamp": df[TIMESTAMP_COLUMN].max(),
        }
        for split_name, df in split_frames.items()
    ]
)

display(split_summary_df)


## Συμβόλαιο Features Και Train-Only Scaling

Η επιλογή features προκύπτει μόνο από το train subset. Τα validation και test χρησιμοποιούνται μόνο για έλεγχο διαθεσιμότητας των train-selected columns και για εφαρμογή του scaler που έχει fit στο train.


In [ ]:
# ============================================================
# NB17 | Train-inferred numeric features και scaling
# ============================================================

candidate_feature_cols = [col for col in train_raw.columns if col not in EXCLUDED_COLUMNS]
numeric_feature_cols = [
    col for col in candidate_feature_cols
    if pd.api.types.is_numeric_dtype(train_raw[col])
]
non_numeric_excluded = sorted(set(candidate_feature_cols) - set(numeric_feature_cols))

if not numeric_feature_cols:
    raise ValueError("Δεν βρέθηκαν train-inferred numeric learned feature columns.")

if len(numeric_feature_cols) != EXPECTED_NUMERIC_FEATURES:
    raise ValueError(
        f"Αναμενόταν {EXPECTED_NUMERIC_FEATURES} numeric features, "
        f"αλλά βρέθηκαν {len(numeric_feature_cols)}."
    )

for blocked in EXCLUDED_COLUMNS:
    if blocked in numeric_feature_cols:
        raise ValueError(f"Blocked column μπήκε στα features: {blocked}")

for split_name, df in split_frames.items():
    missing = [col for col in numeric_feature_cols if col not in df.columns]
    if missing:
        raise KeyError(f"Λείπουν train-selected features από το {split_name}: {missing}")
    selected_nulls = int(df[[TARGET_COLUMN, *numeric_feature_cols]].isnull().sum().sum())
    if selected_nulls:
        raise ValueError(f"Null target/feature values στο {split_name}: {selected_nulls}")
    selected_values = df[[TARGET_COLUMN, *numeric_feature_cols]].to_numpy(dtype=np.float64)
    if not np.isfinite(selected_values).all():
        raise ValueError(f"Βρέθηκαν non-finite target/feature values στο {split_name}.")

scaler = StandardScaler()
train_scaled_values = scaler.fit_transform(train_raw[numeric_feature_cols]).astype(np.float32)
val_scaled_values = scaler.transform(val_raw[numeric_feature_cols]).astype(np.float32)
test_scaled_values = scaler.transform(test_raw[numeric_feature_cols]).astype(np.float32)


def make_scaled_sequence_frame(df: pd.DataFrame, scaled_values: np.ndarray) -> pd.DataFrame:
    out = df[[PARK_ID_COLUMN, TIMESTAMP_COLUMN, TARGET_COLUMN]].copy()
    out[numeric_feature_cols] = scaled_values
    return out


train_scaled_df = make_scaled_sequence_frame(train_raw, train_scaled_values)
val_scaled_df = make_scaled_sequence_frame(val_raw, val_scaled_values)
test_scaled_df = make_scaled_sequence_frame(test_raw, test_scaled_values)

feature_audit_df = pd.DataFrame(
    {
        "feature": numeric_feature_cols,
        "train_dtype": [str(train_raw[col].dtype) for col in numeric_feature_cols],
    }
)

print("Πλήθος numeric learned features:", len(numeric_feature_cols))
print("Non-numeric candidate columns που εξαιρέθηκαν:", non_numeric_excluded or "none")
print("Πολιτική target scaling: ο στόχος παραμένει unscaled.")
display(feature_audit_df.head(20))


## Gap-Safe Κατασκευή Ακολουθιών

Ο constructor εφαρμόζει ρητά τον κανόνα στόχου του NB17. Για `LOOKBACK_STEPS = 24`, κάθε feature tensor χρησιμοποιεί τις γραμμές `[i, ..., i+23]`, ενώ ο στόχος είναι η γραμμή `i+24`. Το target timestamp ελέγχεται ότι βρίσκεται ακριβώς ένα αναμενόμενο βήμα μετά το τελευταίο feature timestamp.


In [ ]:
# ============================================================
# NB17 | Leakage-safe sliding windows μόνο μέσα σε κάθε split
# ============================================================

def build_sequence_split(
    df: pd.DataFrame,
    split_name: str,
    feature_cols: list[str],
    lookback_steps: int,
    expected_freq: pd.Timedelta,
) -> tuple[np.ndarray, np.ndarray, pd.DataFrame, pd.DataFrame]:
    X_parts: list[np.ndarray] = []
    y_parts: list[float] = []
    meta_rows: list[dict[str, Any]] = []
    audit_rows: list[dict[str, Any]] = []

    sorted_df = df.sort_values([PARK_ID_COLUMN, TIMESTAMP_COLUMN], kind="mergesort").reset_index(drop=True)

    for park_id, park_df in sorted_df.groupby(PARK_ID_COLUMN, sort=False):
        park_df = park_df.sort_values(TIMESTAMP_COLUMN, kind="mergesort").reset_index(drop=True)
        gap_start = park_df[TIMESTAMP_COLUMN].diff().ne(expected_freq)
        segment_ids = gap_start.cumsum()

        for segment_id, segment_df in park_df.groupby(segment_ids, sort=False):
            segment_df = segment_df.reset_index(drop=True)
            segment_len = len(segment_df)
            n_windows = max(segment_len - lookback_steps, 0)

            audit_rows.append(
                {
                    "split": split_name,
                    "park_id": park_id,
                    "segment_id": int(segment_id),
                    "segment_start_timestamp": segment_df[TIMESTAMP_COLUMN].iloc[0],
                    "segment_end_timestamp": segment_df[TIMESTAMP_COLUMN].iloc[-1],
                    "segment_rows": segment_len,
                    "lookback_steps": lookback_steps,
                    "windows": n_windows,
                }
            )

            if n_windows == 0:
                continue

            feature_values = segment_df[feature_cols].to_numpy(dtype=np.float32)
            target_values = segment_df[TARGET_COLUMN].to_numpy(dtype=np.float32)
            timestamps = segment_df[TIMESTAMP_COLUMN].reset_index(drop=True)

            for start_idx in range(n_windows):
                feature_start_idx = start_idx
                feature_end_exclusive = start_idx + lookback_steps
                target_idx = start_idx + lookback_steps

                window_start_ts = timestamps.iloc[feature_start_idx]
                window_end_ts = timestamps.iloc[feature_end_exclusive - 1]
                target_ts = timestamps.iloc[target_idx]

                if target_ts - window_end_ts != expected_freq:
                    raise ValueError(
                        f"Το target timestamp δεν είναι η επόμενη γραμμή μετά το feature window στο {split_name}, "
                        f"park={park_id}, segment={segment_id}."
                    )
                if target_ts in set(timestamps.iloc[feature_start_idx:feature_end_exclusive]):
                    raise ValueError("Το target timestamp διέρρευσε μέσα στο feature window.")

                X_parts.append(feature_values[feature_start_idx:feature_end_exclusive])
                y_parts.append(float(target_values[target_idx]))
                meta_rows.append(
                    {
                        "split": split_name,
                        "park_id": park_id,
                        "segment_id": int(segment_id),
                        "window_start_timestamp": window_start_ts,
                        "window_end_timestamp": window_end_ts,
                        "target_timestamp": target_ts,
                    }
                )

    if X_parts:
        X = np.stack(X_parts).astype(np.float32)
        y = np.asarray(y_parts, dtype=np.float32)
    else:
        X = np.empty((0, lookback_steps, len(feature_cols)), dtype=np.float32)
        y = np.empty((0,), dtype=np.float32)

    meta_df = pd.DataFrame(meta_rows)
    audit_df = pd.DataFrame(audit_rows)
    return X, y, meta_df, audit_df


X_train_seq, y_train_seq, train_window_meta_df, train_window_audit_df = build_sequence_split(
    train_scaled_df, "train", numeric_feature_cols, LOOKBACK_STEPS, EXPECTED_FREQ
)
X_val_seq, y_val_seq, val_window_meta_df, val_window_audit_df = build_sequence_split(
    val_scaled_df, "validation", numeric_feature_cols, LOOKBACK_STEPS, EXPECTED_FREQ
)
X_test_seq, y_test_seq, test_window_meta_df, test_window_audit_df = build_sequence_split(
    test_scaled_df, "test", numeric_feature_cols, LOOKBACK_STEPS, EXPECTED_FREQ
)

window_audit_df = pd.concat(
    [train_window_audit_df, val_window_audit_df, test_window_audit_df],
    ignore_index=True,
)

sequence_summary_df = pd.DataFrame(
    [
        {"split": "train", "windows": len(X_train_seq), "shape": X_train_seq.shape},
        {"split": "validation", "windows": len(X_val_seq), "shape": X_val_seq.shape},
        {"split": "test", "windows": len(X_test_seq), "shape": X_test_seq.shape},
    ]
)

display(sequence_summary_df)
display(window_audit_df.head(12))


In [ ]:
# ============================================================
# NB17 | Smoke caps και dataloaders
# ============================================================

def cap_sequence_arrays(
    X: np.ndarray,
    y: np.ndarray,
    meta_df: pd.DataFrame,
    split_name: str,
    smoke_mode: bool,
) -> tuple[np.ndarray, np.ndarray, pd.DataFrame]:
    if not smoke_mode:
        return X, y, meta_df

    cap = SMOKE_MAX_WINDOWS[split_name]
    capped_n = min(cap, len(X))
    return X[:capped_n], y[:capped_n], meta_df.iloc[:capped_n].reset_index(drop=True)


X_train_model, y_train_model, train_model_meta_df = cap_sequence_arrays(
    X_train_seq, y_train_seq, train_window_meta_df, "train", SMOKE_MODE
)
X_val_model, y_val_model, val_model_meta_df = cap_sequence_arrays(
    X_val_seq, y_val_seq, val_window_meta_df, "validation", SMOKE_MODE
)
X_test_model, y_test_model, test_model_meta_df = cap_sequence_arrays(
    X_test_seq, y_test_seq, test_window_meta_df, "test", SMOKE_MODE
)


def make_loader(X: np.ndarray, y: np.ndarray, batch_size: int, shuffle: bool, seed: int) -> DataLoader:
    dataset = TensorDataset(
        torch.from_numpy(X.astype(np.float32)),
        torch.from_numpy(y.astype(np.float32).reshape(-1, 1)),
    )
    generator = None
    if shuffle:
        generator = torch.Generator()
        generator.manual_seed(seed)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, generator=generator)


if min(len(X_train_model), len(X_val_model), len(X_test_model)) == 0:
    raise ValueError("At least one split has no sequence windows after smoke/full selection.")

train_loader = make_loader(X_train_model, y_train_model, BATCH_SIZE, shuffle=True, seed=SEED)
val_loader = make_loader(X_val_model, y_val_model, BATCH_SIZE, shuffle=False, seed=SEED)
test_loader = make_loader(X_test_model, y_test_model, BATCH_SIZE, shuffle=False, seed=SEED)

model_window_summary_df = pd.DataFrame(
    [
        {"split": "train", "model_windows": len(X_train_model), "source_windows": len(X_train_seq)},
        {"split": "validation", "model_windows": len(X_val_model), "source_windows": len(X_val_seq)},
        {"split": "test", "model_windows": len(X_test_model), "source_windows": len(X_test_seq)},
    ]
)

display(model_window_summary_df)


In [ ]:
# ============================================================
# NB17 | Προαιρετικό import του Mamba block
# ============================================================

try:
    from mamba_ssm import Mamba

    MAMBA_AVAILABLE = True
    MAMBA_IMPORT_ERROR = None
except Exception as exc:
    Mamba = None
    MAMBA_AVAILABLE = False
    MAMBA_IMPORT_ERROR = repr(exc)

if MAMBA_AVAILABLE:
    print("Το mamba_ssm είναι διαθέσιμο. Mamba smoke training μπορεί να ενεργοποιηθεί μόνο αν οριστεί ρητά.")
else:
    print("Το mamba_ssm δεν είναι διαθέσιμο σε αυτό το περιβάλλον.")
    print('Για Colab/GPU run, χρησιμοποιήστε χειροκίνητα: pip install "mamba-ssm[causal-conv1d]" --no-build-isolation')
    print("Το notebook συνεχίζει χωρίς αποτυχία και δεν εκπαιδεύει από προεπιλογή.")
    print("Λεπτομέρεια import:", MAMBA_IMPORT_ERROR)


## Ορισμός Mamba Μοντέλου

Το μοντέλο δέχεται input tensor σχήματος `[batch, LOOKBACK_STEPS, n_features]`, εφαρμόζει γραμμική προβολή από τα 41 numeric features σε μικρό `d_model`, περνά ένα μικρό Mamba block stack και χρησιμοποιεί last-step pooling πριν από scalar regression head. Το μέγεθος είναι σκόπιμα μικρό για εφικτό Colab smoke run.


In [ ]:
# ============================================================
# NB17 | Mamba model, metrics και training helpers
# ============================================================

class MambaRegressor(nn.Module):
    def __init__(
        self,
        input_dim: int,
        d_model: int,
        n_layers: int,
        d_state: int,
        d_conv: int,
        expand: int,
        dropout: float,
    ) -> None:
        super().__init__()
        if Mamba is None:
            raise ImportError("Το mamba_ssm δεν είναι διαθέσιμο. Δεν μπορεί να αρχικοποιηθεί MambaRegressor.")

        self.input_projection = nn.Linear(input_dim, d_model)
        self.blocks = nn.ModuleList(
            [
                Mamba(
                    d_model=d_model,
                    d_state=d_state,
                    d_conv=d_conv,
                    expand=expand,
                )
                for _ in range(n_layers)
            ]
        )
        self.norms = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(n_layers)])
        self.dropout = nn.Dropout(dropout)
        head_hidden = max(d_model // 2, 8)
        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, head_hidden),
            nn.GELU(),
            nn.Linear(head_hidden, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        hidden = self.input_projection(x)
        for block, norm in zip(self.blocks, self.norms):
            hidden = norm(hidden + block(hidden))
            hidden = self.dropout(hidden)
        last_step = hidden[:, -1, :]
        return self.head(last_step)


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    residual = y_true - y_pred
    mae = float(np.mean(np.abs(residual)))
    rmse = float(np.sqrt(np.mean(residual**2)))
    ss_res = float(np.sum(residual**2))
    ss_tot = float(np.sum((y_true - np.mean(y_true)) ** 2))
    r2 = float(1.0 - ss_res / ss_tot) if ss_tot > 0 else np.nan
    return {"MAE": mae, "RMSE": rmse, "R2": r2}


def run_epoch(model: nn.Module, loader: DataLoader, criterion: nn.Module, optimizer: Any | None = None) -> float:
    is_train = optimizer is not None
    model.train(mode=is_train)
    total_loss = 0.0
    total_count = 0

    for xb, yb in loader:
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)

        if is_train:
            optimizer.zero_grad()

        pred = model(xb)
        loss = criterion(pred, yb)

        if is_train:
            loss.backward()
            optimizer.step()

        batch_size_current = xb.shape[0]
        total_loss += float(loss.detach().cpu().item()) * batch_size_current
        total_count += batch_size_current

    if total_count == 0:
        raise ValueError("Δεν μπορεί να τρέξει epoch σε empty loader.")
    return total_loss / total_count


@torch.no_grad()
def predict_loader(model: nn.Module, loader: DataLoader) -> np.ndarray:
    model.eval()
    preds: list[np.ndarray] = []
    for xb, _ in loader:
        xb = xb.to(DEVICE)
        pred = model(xb).detach().cpu().numpy().reshape(-1)
        preds.append(pred)
    if not preds:
        raise ValueError("Δεν μπορεί να γίνει prediction από empty loader.")
    return np.concatenate(preds).astype(float)


def train_mamba_with_early_stopping(
    train_loader: DataLoader,
    val_loader: DataLoader,
    input_dim: int,
    epochs: int,
    patience: int,
) -> tuple[nn.Module, pd.DataFrame, int, float]:
    set_reproducibility(SEED)
    model = MambaRegressor(
        input_dim=input_dim,
        d_model=D_MODEL,
        n_layers=MAMBA_LAYERS,
        d_state=MAMBA_D_STATE,
        d_conv=MAMBA_D_CONV,
        expand=MAMBA_EXPAND,
        dropout=DROPOUT,
    ).to(DEVICE)
    criterion = nn.MSELoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

    val_targets = np.concatenate(
        [yb.detach().cpu().numpy().reshape(-1) for _, yb in val_loader]
    ).astype(float)

    best_state = None
    best_epoch = 0
    best_val_loss = float("inf")
    best_selection_key: tuple[float, float, float] | None = None
    epochs_without_improvement = 0
    history_rows: list[dict[str, Any]] = []

    for epoch in range(1, epochs + 1):
        train_loss = run_epoch(model, train_loader, criterion, optimizer=optimizer)
        val_loss = run_epoch(model, val_loader, criterion, optimizer=None)
        val_pred = predict_loader(model, val_loader)
        val_metrics = compute_metrics(val_targets, val_pred)
        selection_key = (val_metrics["MAE"], val_metrics["RMSE"], -val_metrics["R2"])
        improved = best_selection_key is None or selection_key < best_selection_key

        if improved:
            best_selection_key = selection_key
            best_val_loss = val_loss
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        history_rows.append(
            {
                "run_mode": "smoke" if SMOKE_MODE else "subset",
                "evidence_status": "έλεγχος κώδικα μόνο, όχι manuscript evidence" if SMOKE_MODE else "subset evidence μόνο, όχι αντικατάσταση benchmark",
                "epoch": epoch,
                "train_loss_mse": train_loss,
                "val_loss_mse": val_loss,
                "val_MAE": val_metrics["MAE"],
                "val_RMSE": val_metrics["RMSE"],
                "val_R2": val_metrics["R2"],
                "is_best_epoch": improved,
            }
        )

        if epochs_without_improvement >= patience:
            break

    if best_state is None:
        raise RuntimeError("Δεν κρατήθηκε validation-selected mamba state στη μνήμη.")

    model.load_state_dict(best_state)
    return model, pd.DataFrame(history_rows), best_epoch, float(best_val_loss)

print("MambaRegressor ορίστηκε. Input shape: [batch, LOOKBACK_STEPS, n_features].")


## Smoke-Safe Εκπαίδευση

Το `RUN_SMOKE_TRAINING` είναι προεπιλεγμένα `False`, άρα η εκτέλεση του notebook δεν εκπαιδεύει μοντέλο. Αν οριστεί `True`, το `SMOKE_MODE` περιορίζει windows και epochs ώστε το run να είναι μόνο έλεγχος κώδικα και όχι manuscript evidence.


In [ ]:
# ============================================================
# NB17 | Validation-only επιλογή Mamba μοντέλου
# ============================================================

selected_model: nn.Module | None = None
training_history_df = pd.DataFrame()
validation_metrics_df = pd.DataFrame()
best_epoch: int | None = None
best_val_loss_mse: float | None = None

if not RUN_SMOKE_TRAINING:
    print("RUN_SMOKE_TRAINING είναι False· δεν εκτελέστηκε Mamba training.")
    print("Ορίστε RUN_SMOKE_TRAINING = True μόνο για μικρό smoke validation, όχι για manuscript evidence.")
elif not MAMBA_AVAILABLE:
    print("Το mamba_ssm δεν είναι διαθέσιμο. Παραλείπεται το Mamba training.")
    print("Χρειάζεται χειροκίνητη εγκατάσταση σε Colab/GPU πριν από οποιοδήποτε training run.")
else:
    run_start = time.perf_counter()
    effective_epochs = SMOKE_EPOCHS if SMOKE_MODE else EPOCHS
    effective_patience = SMOKE_PATIENCE if SMOKE_MODE else PATIENCE

    selected_model, training_history_df, best_epoch, best_val_loss_mse = train_mamba_with_early_stopping(
        train_loader=train_loader,
        val_loader=val_loader,
        input_dim=len(numeric_feature_cols),
        epochs=effective_epochs,
        patience=effective_patience,
    )

    val_pred = predict_loader(selected_model, val_loader)
    val_metrics = compute_metrics(y_val_model, val_pred)
    elapsed_seconds = time.perf_counter() - run_start

    validation_metrics_df = pd.DataFrame(
        [
            {
                "run_mode": "smoke" if SMOKE_MODE else "subset",
                "evidence_status": "έλεγχος κώδικα μόνο, όχι manuscript evidence" if SMOKE_MODE else "subset evidence μόνο, όχι αντικατάσταση benchmark",
                "model": "PyTorch Mamba sequence baseline",
                "selected_parks": ";".join(SELECTED_PARKS),
                "n_parks": len(SELECTED_PARKS),
                "seed": SEED,
                "device": str(DEVICE),
                "lookback_steps": LOOKBACK_STEPS,
                "d_model": D_MODEL,
                "mamba_layers": MAMBA_LAYERS,
                "mamba_d_state": MAMBA_D_STATE,
                "mamba_d_conv": MAMBA_D_CONV,
                "mamba_expand": MAMBA_EXPAND,
                "dropout": DROPOUT,
                "learning_rate": LEARNING_RATE,
                "weight_decay": WEIGHT_DECAY,
                "batch_size": BATCH_SIZE,
                "epochs_requested": effective_epochs,
                "best_epoch": best_epoch,
                "best_val_loss_mse": best_val_loss_mse,
                "n_numeric_features": len(numeric_feature_cols),
                "train_windows_used": len(X_train_model),
                "val_windows_used": len(X_val_model),
                "test_windows_available": len(X_test_model),
                "MAE": val_metrics["MAE"],
                "RMSE": val_metrics["RMSE"],
                "R2": val_metrics["R2"],
                "selection_split": "validation",
                "selection_policy": "best epoch με βάση validation MAE, μετά RMSE, μετά R2",
                "elapsed_seconds": round(elapsed_seconds, 3),
            }
        ]
    )

    display(training_history_df)
    display(validation_metrics_df)


## Πύλη Εφάπαξ Test Evaluation

Το `RUN_TEST_EVALUATION` είναι προεπιλεγμένα `False`. Αν ενεργοποιηθεί αφού υπάρχει validation-selected in-memory Mamba state, το επιλεγμένο state αξιολογείται μία φορά στο test subset. Δεν γίνεται test-driven model selection.


In [ ]:
# ============================================================
# NB17 | Πύλη τελικής test-only αξιολόγησης
# ============================================================

selected_test_metrics_df = pd.DataFrame()
test_predictions_residuals_df = pd.DataFrame()
test_evaluations = 0

if RUN_TEST_EVALUATION and selected_model is not None:
    test_pred = predict_loader(selected_model, test_loader)
    test_metrics = compute_metrics(y_test_model, test_pred)
    test_evaluations = 1

    selected_test_metrics_df = pd.DataFrame(
        [
            {
                "run_mode": "smoke" if SMOKE_MODE else "subset",
                "evidence_status": "έλεγχος κώδικα μόνο, όχι manuscript evidence" if SMOKE_MODE else "subset evidence μόνο, όχι αντικατάσταση benchmark",
                "model": "PyTorch Mamba sequence baseline",
                "selected_parks": ";".join(SELECTED_PARKS),
                "n_parks": len(SELECTED_PARKS),
                "seed": SEED,
                "device": str(DEVICE),
                "lookback_steps": LOOKBACK_STEPS,
                "d_model": D_MODEL,
                "mamba_layers": MAMBA_LAYERS,
                "learning_rate": LEARNING_RATE,
                "weight_decay": WEIGHT_DECAY,
                "batch_size": BATCH_SIZE,
                "epochs_requested": SMOKE_EPOCHS if SMOKE_MODE else EPOCHS,
                "best_epoch": best_epoch,
                "best_val_loss_mse": best_val_loss_mse,
                "n_numeric_features": len(numeric_feature_cols),
                "train_windows_used": len(X_train_model),
                "val_windows_used": len(X_val_model),
                "test_windows_used": len(X_test_model),
                "MAE": test_metrics["MAE"],
                "RMSE": test_metrics["RMSE"],
                "R2": test_metrics["R2"],
                "test_evaluations": test_evaluations,
                "test_policy": "το validation-selected state αξιολογείται μία φορά στο test subset",
            }
        ]
    )

    test_predictions_residuals_df = test_model_meta_df.copy()
    test_predictions_residuals_df["y_true"] = y_test_model.astype(float)
    test_predictions_residuals_df["y_pred"] = test_pred.astype(float)
    test_predictions_residuals_df["residual"] = test_predictions_residuals_df["y_true"] - test_predictions_residuals_df["y_pred"]

    display(selected_test_metrics_df)
elif RUN_TEST_EVALUATION:
    print("Το RUN_TEST_EVALUATION είναι True αλλά δεν υπάρχει validation-selected Mamba model. Δεν έγινε test evaluation.")
else:
    print("Το RUN_TEST_EVALUATION είναι False. Το test split δεν αξιολογήθηκε.")


## Προαιρετικά Local-Only Exports

Τα exports είναι απενεργοποιημένα από προεπιλογή. Αν ενεργοποιηθούν αργότερα, γράφονται μόνο τα paths που δηλώνονται σε αυτό το notebook και μόνο κάτω από `data/processed/diagnostics/nn_sequence_subset_mamba/`.


In [ ]:
# ============================================================
# NB17 | Προαιρετικά local-only CSV exports, απενεργοποιημένα από προεπιλογή
# ============================================================

exported_csv_paths: list[Path] = []

run_manifest_df = pd.DataFrame(
    [
        {"field": "notebook", "value": "notebooks/17_mamba_sequence_baseline_subset.ipynb"},
        {"field": "evidence_status", "value": "έλεγχος κώδικα μόνο, όχι manuscript evidence" if SMOKE_MODE else "subset evidence μόνο, όχι αντικατάσταση benchmark"},
        {"field": "selected_parks", "value": ";".join(SELECTED_PARKS)},
        {"field": "target_column", "value": TARGET_COLUMN},
        {"field": "excluded_columns", "value": ", ".join(sorted(EXCLUDED_COLUMNS))},
        {"field": "feature_contract", "value": "μόνο numeric learned train columns, χωρίς turbine encoding και χωρίς validation/test statistics για feature selection"},
        {"field": "scaling_policy", "value": "StandardScaler fit μόνο σε train subset feature rows, target unscaled"},
        {"field": "sequence_policy", "value": "X rows [i..i+23] προβλέπουν target row i+24 για LOOKBACK_STEPS=24"},
        {"field": "selection_policy", "value": "best in-memory Mamba state με βάση validation MAE, με RMSE και R2 ως tie-breakers"},
        {"field": "test_policy", "value": "το validation-selected state αξιολογείται μία φορά στο test μόνο αν RUN_TEST_EVALUATION=True"},
        {"field": "baseline_metrics_policy", "value": "το data/processed/baseline_metrics.csv δεν τροποποιείται"},
        {"field": "model_artifact_policy", "value": "δεν γράφονται checkpoints ή model binaries"},
        {"field": "mamba_available", "value": MAMBA_AVAILABLE},
        {"field": "export_results", "value": EXPORT_RESULTS},
    ]
)

if EXPORT_RESULTS:
    output_dir_resolved = OUTPUT_DIR.resolve()
    if output_dir_resolved != EXPECTED_OUTPUT_DIR.resolve():
        raise ValueError(
            "Το OUTPUT_DIR δεν δείχνει στον reserved diagnostics φάκελο."
        )

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    run_manifest_df.to_csv(RUN_MANIFEST_PATH, index=False)
    window_audit_df.to_csv(WINDOW_AUDIT_PATH, index=False)
    training_history_df.to_csv(TRAINING_HISTORY_PATH, index=False)
    validation_metrics_df.to_csv(VALIDATION_METRICS_PATH, index=False)
    selected_test_metrics_df.to_csv(SELECTED_TEST_METRICS_PATH, index=False)
    test_predictions_residuals_df.to_csv(TEST_PREDICTIONS_RESIDUALS_PATH, index=False)

    exported_csv_paths = [
        RUN_MANIFEST_PATH,
        WINDOW_AUDIT_PATH,
        TRAINING_HISTORY_PATH,
        VALIDATION_METRICS_PATH,
        SELECTED_TEST_METRICS_PATH,
        TEST_PREDICTIONS_RESIDUALS_PATH,
    ]
    print("Γράφτηκαν local-only NB17 exports:")
    for path in exported_csv_paths:
        print("-", path)
else:
    print("Το EXPORT_RESULTS είναι False. Δεν γράφτηκαν generated CSV outputs.")


In [ ]:
# ============================================================
# NB17 | Τελική σύνοψη ελέγχων
# ============================================================

expected_output_dir = EXPECTED_OUTPUT_DIR

self_check_rows = [
    {"check": "seed_default", "status": SEED == 42, "detail": SEED},
    {"check": "smoke_mode_default", "status": SMOKE_MODE is True, "detail": SMOKE_MODE},
    {"check": "run_smoke_training_default", "status": RUN_SMOKE_TRAINING is False, "detail": RUN_SMOKE_TRAINING},
    {"check": "run_test_evaluation_default", "status": RUN_TEST_EVALUATION is False, "detail": RUN_TEST_EVALUATION},
    {"check": "export_results_default", "status": EXPORT_RESULTS is False, "detail": EXPORT_RESULTS},
    {"check": "selected_parks", "status": SELECTED_PARKS == ["00183", "00198", "00303", "00427"], "detail": ";".join(SELECTED_PARKS)},
    {"check": "lookback_steps", "status": LOOKBACK_STEPS == 24, "detail": LOOKBACK_STEPS},
    {"check": "numeric_feature_count", "status": len(numeric_feature_cols) == 41, "detail": len(numeric_feature_cols)},
    {"check": "target_unscaled", "status": True, "detail": TARGET_COLUMN},
    {"check": "mamba_only_model_class", "status": "MambaRegressor" in globals(), "detail": "Ορίζεται μόνο Mamba scaffold."},
    {"check": "no_test_evaluation_by_default", "status": test_evaluations == 0, "detail": test_evaluations},
    {"check": "no_exports_by_default", "status": len(exported_csv_paths) == 0, "detail": len(exported_csv_paths)},
    {"check": "reserved_output_dir", "status": OUTPUT_DIR == expected_output_dir, "detail": str(OUTPUT_DIR)},
    {"check": "baseline_metrics_not_written", "status": True, "detail": str(BASELINE_METRICS_PATH)},
]

self_check_df = pd.DataFrame(self_check_rows)
display(self_check_df)

if not self_check_df["status"].astype(bool).all():
    raise ValueError("Απέτυχε ένας ή περισσότεροι τελικοί έλεγχοι του NB17 scaffold.")

print("Ο τελικός έλεγχος του NB17 scaffold πέρασε.")
print("Checklist: ασφαλή defaults, parks 00183/00198/00303/00427, lookback 24, test evaluations 0, exports 0.")
print("Checkpoint/model binary outputs που γράφτηκαν: 0")
